# Complex Land / Address Geolocation — SageMaker Pipeline

This notebook uses exactly the data you have:

- **OS AddressBase Core** — CSV
- **OS Open Names** — CSV
- **OS Open UPRN** — CSV or GeoPackage
- **OS Open Roads** — GeoPackage
- **HMLR INSPIRE** — GML under local-authority folders

Expected S3 layout:

```text
s3://YOUR-BUCKET/
raw/os/addressbase_core/
raw/os/open_names/
raw/os/open_uprn/
raw/os/open_roads/
raw/hmlr/la/

processed/os/address_records/
processed/os/address_tokens/
processed/os/name_records/
processed/os/name_tokens/
processed/os/uprn_records/
processed/os/roads/
processed/hmlr/inspire/
results/
```

The runtime is **text-first and fully automated**. It keeps multiple candidates for ambiguous anchors, combines them as geographic hypotheses, evaluates nearby HMLR INSPIRE polygons with deterministic GIS, then uses Azure OpenAI only for linguistic parsing and evidence adjudication. Weak evidence remains `MULTIPLE_CANDIDATES` or `UNRESOLVED`.

**HMLR INSPIRE is indicative freehold extent data and must not be treated as a definitive legal title boundary.**

In [ ]:
# Install in SageMaker. Restart kernel if requested.
%pip install -q -U "openai>=2.0.0" "pydantic>=2.7" boto3 s3fs pyarrow pandas numpy rapidfuzz geopandas shapely pyproj fiona

In [ ]:
from __future__ import annotations

import os, io, re, json, math, uuid, hashlib, tempfile, logging, unicodedata
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Optional, Literal, Iterable

import boto3
import numpy as np
import pandas as pd
import geopandas as gpd
import fiona
from rapidfuzz import fuzz
from shapely.geometry import Point, LineString, MultiPoint, box, mapping, shape
from shapely.ops import nearest_points, unary_union
from shapely.validation import make_valid
from pydantic import BaseModel, Field, ConfigDict
try:
    from openai import OpenAI
except Exception:
    OpenAI = Any  # allows offline deterministic tests before installing openai>=2

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('land-geolocator')

## Config

In [ ]:
# --------------------------- CONFIG ---------------------------
@dataclass
class RawPaths:
    addressbase_core: str = 's3://YOUR-BUCKET/raw/os/addressbase_core/'
    open_names: str = 's3://YOUR-BUCKET/raw/os/open_names/'
    open_uprn: str = 's3://YOUR-BUCKET/raw/os/open_uprn/'
    open_roads: str = 's3://YOUR-BUCKET/raw/os/open_roads/'
    hmlr_inspire: str = 's3://YOUR-BUCKET/raw/hmlr/la/'

@dataclass
class ProcessedPaths:
    address_records: str = 's3://YOUR-BUCKET/processed/os/address_records/'
    address_tokens: str = 's3://YOUR-BUCKET/processed/os/address_tokens/'
    name_records: str = 's3://YOUR-BUCKET/processed/os/name_records/'
    name_tokens: str = 's3://YOUR-BUCKET/processed/os/name_tokens/'
    uprn_records: str = 's3://YOUR-BUCKET/processed/os/uprn_records/'
    road_geometries: str = 's3://YOUR-BUCKET/processed/os/roads/'
    inspire: str = 's3://YOUR-BUCKET/processed/hmlr/inspire/'
    results: str = 's3://YOUR-BUCKET/results/'

@dataclass
class PreprocessConfig:
    csv_chunksize: int = 250_000
    parquet_compression: str = 'zstd'
    uprn_bucket_count: int = 256
    name_bucket_count: int = 128
    token_prefix_chars: int = 2
    compact_after_processing: bool = True

@dataclass
class RetrievalConfig:
    address_top_k: int = 20
    name_top_k: int = 20
    max_query_tokens: int = 5
    max_postings_per_token: int = 120_000
    max_address_candidate_pool: int = 5_000
    max_name_candidate_pool: int = 3_000
    min_address_score: float = 58.0
    min_name_score: float = 58.0
    uprn_coordinate_warning_m: float = 25.0
    uprn_coordinate_hard_disagreement_m: float = 100.0
    beam_width: int = 24
    max_candidates_per_anchor_for_beam: int = 12
    hypothesis_distance_scale_m: float = 2_500.0
    hard_anchor_separation_m: float = 30_000.0
    single_property_radius_m: float = 350.0
    single_road_radius_m: float = 900.0
    single_settlement_radius_m: float = 3_000.0
    default_search_radius_m: float = 600.0
    max_search_radius_m: float = 3_500.0
    hmlr_candidate_limit_per_hypothesis: int = 300

@dataclass
class SpatialConfig:
    anchor_polygon_lookup_tolerance_m: float = 8.0
    adjacent_tolerance_m: float = 8.0
    near_distance_m: float = 100.0
    boundary_tolerance_m: float = 15.0
    between_corridor_m: float = 35.0
    directional_tolerance_deg: float = 55.0
    rear_front_max_distance_m: float = 150.0
    opposite_max_distance_m: float = 130.0
    junction_distance_m: float = 90.0
    required_constraint_min_score: float = 0.25
    required_constraint_penalty: float = 0.25
    relation_weights: dict = field(default_factory=lambda: {
        'at': 2.2, 'bounded_by': 2.0, 'adjacent_to': 1.9, 'between': 1.6,
        'rear_of': 1.5, 'front_of': 1.3, 'opposite': 1.4, 'on_side_of': 1.4,
        'at_junction_of': 1.6, 'north_of': 1.0, 'south_of': 1.0,
        'east_of': 1.0, 'west_of': 1.0, 'near': 0.6,
        'accessed_from': 0.8, 'part_of': 1.2,
    })

@dataclass
class DecisionConfig:
    max_hypotheses_to_score: int = 8
    max_candidates_for_llm: int = 10
    spatial_weight: float = 0.68
    hypothesis_weight: float = 0.20
    semantic_weight: float = 0.12
    matched_threshold: float = 0.79
    multiple_threshold: float = 0.48
    minimum_margin: float = 0.10
    minimum_constraint_coverage_for_match: float = 0.70
    minimum_anchor_resolution_for_match: float = 0.70

@dataclass
class LLMConfig:
    parse_twice: bool = True
    use_semantic_adjudicator: bool = True
    max_search_variants_per_anchor: int = 5

@dataclass
class AppConfig:
    aws_region: str = os.getenv('AWS_REGION', 'eu-west-2')
    azure_endpoint: str = os.getenv('AZURE_OPENAI_ENDPOINT', 'https://YOUR-RESOURCE-NAME.openai.azure.com')
    azure_api_key: str = os.getenv('AZURE_OPENAI_API_KEY', '')
    azure_deployment: str = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'YOUR_DEPLOYMENT_NAME')
    raw: RawPaths = field(default_factory=RawPaths)
    processed: ProcessedPaths = field(default_factory=ProcessedPaths)
    preprocess: PreprocessConfig = field(default_factory=PreprocessConfig)
    retrieval: RetrievalConfig = field(default_factory=RetrievalConfig)
    spatial: SpatialConfig = field(default_factory=SpatialConfig)
    decision: DecisionConfig = field(default_factory=DecisionConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)

## Text / Ids

In [ ]:
# --------------------------- TEXT / IDS ---------------------------
STOPWORDS = {'the', 'of', 'and', 'at', 'in', 'on', 'to'}

def normalize_text(value: Optional[str]) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = unicodedata.normalize('NFKD', str(value).casefold())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def tokenize(value: Optional[str]) -> list[str]:
    return list(dict.fromkeys(t for t in normalize_text(value).split() if t not in STOPWORDS and t))

def token_prefix(token: str, chars: int) -> str:
    t = normalize_text(token).replace(' ', '')
    return (t[:chars] if t else '_').ljust(chars, '_')

def stable_bucket(value: str, count: int) -> int:
    digest = hashlib.blake2b(str(value).encode('utf-8'), digest_size=8).digest()
    return int.from_bytes(digest, 'big') % count

def address_query_variants(text: str) -> list[str]:
    text = re.sub(r'\s+', ' ', text.strip())
    variants = [text]
    m = re.search(r'\b(\d{1,4})(?:\s*[-–—]\s*|\s+to\s+)(\d{1,4})\b', text, flags=re.I)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        if abs(b - a) <= 20:
            lo, hi = sorted((a, b))
            for n in range(lo, hi + 1):
                variants.append(text[:m.start()] + str(n) + text[m.end():])
    return list(dict.fromkeys(v.strip() for v in variants if v.strip()))

def build_token_rows_vectorized(ids: pd.Series, text: pd.Series, id_col: str, prefix_chars: int) -> pd.DataFrame:
    work = pd.DataFrame({id_col: ids.astype('string'), '_text': text.fillna('').map(normalize_text)})
    ex = work.assign(token=work['_text'].str.split()).explode('token')
    ex = ex[ex['token'].notna() & (~ex['token'].isin(STOPWORDS)) & (ex['token'].str.len() > 0)][[id_col, 'token']]
    ex = ex.drop_duplicates([id_col, 'token'])
    ex['token_prefix'] = ex['token'].map(lambda t: token_prefix(t, prefix_chars))
    return ex

## Schema Maps

In [ ]:
# --------------------------- SCHEMA MAPS ---------------------------
def find_column(columns: Iterable[str], candidates: list[str], required: bool = True) -> Optional[str]:
    cols = list(columns); lower = {str(c).strip().lower(): c for c in cols}
    for c in candidates:
        if c.lower() in lower: return lower[c.lower()]
    for c in candidates:
        for col in cols:
            if c.lower() in str(col).lower(): return col
    if required: raise KeyError(f'Could not find any of {candidates}. Available columns: {cols}')
    return None

def map_columns(columns: Iterable[str], required_map: dict, optional_map: Optional[dict] = None) -> dict:
    out = {target: find_column(columns, candidates, True) for target, candidates in required_map.items()}
    for target, candidates in (optional_map or {}).items(): out[target] = find_column(columns, candidates, False)
    return out

ADDRESSBASE_REQUIRED = {
    'uprn': ['UPRN','uprn'], 'full_address': ['SINGLE_LINE_ADDRESS','single_line_address','FULL_ADDRESS','full_address'],
    'postcode': ['POSTCODE','postcode'], 'town': ['TOWN_NAME','town_name','POST_TOWN','post_town'],
    'easting': ['X_COORDINATE','x_coordinate','EASTING','easting'], 'northing': ['Y_COORDINATE','y_coordinate','NORTHING','northing'],
}
ADDRESSBASE_OPTIONAL = {
    'classification_code': ['CLASSIFICATION_CODE','classification_code'], 'latitude': ['LATITUDE','latitude'],
    'longitude': ['LONGITUDE','longitude'], 'usrn': ['USRN','usrn'], 'toid': ['TOID','toid'],
}
OPEN_NAMES_REQUIRED = {
    'name_id': ['ID','id','NAMES_URI','names_uri','identifier'], 'name': ['NAME1','name1','NAME','name'],
    'feature_type': ['LOCAL_TYPE','local_type','TYPE','type'], 'easting': ['GEOMETRY_X','geometry_x','EASTING','easting','X','x'],
    'northing': ['GEOMETRY_Y','geometry_y','NORTHING','northing','Y','y'],
}
OPEN_UPRN_REQUIRED = {
    'uprn': ['UPRN','uprn'], 'easting': ['X_COORDINATE','x_coordinate'], 'northing': ['Y_COORDINATE','y_coordinate'],
    'latitude': ['LATITUDE','latitude'], 'longitude': ['LONGITUDE','longitude'],
}

## S3 / Preprocessing

In [ ]:
# --------------------------- S3 / PREPROCESSING ---------------------------
def s3_client_for(config: AppConfig):
    return boto3.client('s3', region_name=config.aws_region)

def parse_s3_uri(uri: str) -> tuple[str, str]:
    if not uri.startswith('s3://'): raise ValueError(f'Expected S3 URI: {uri}')
    rest = uri[5:]; bucket, _, key = rest.partition('/'); return bucket, key

def list_s3_files(prefix: str, suffixes: tuple[str, ...], client=None) -> list[str]:
    client = client or boto3.client('s3')
    bucket, key_prefix = parse_s3_uri(prefix.rstrip('/') + '/')
    paginator = client.get_paginator('list_objects_v2'); out=[]
    for page in paginator.paginate(Bucket=bucket, Prefix=key_prefix):
        for item in page.get('Contents', []):
            key=item['Key']
            if key.lower().endswith(tuple(s.lower() for s in suffixes)): out.append(f's3://{bucket}/{key}')
    return sorted(out)

def delete_s3_prefix(prefix: str, client=None):
    client = client or boto3.client('s3'); bucket, key_prefix = parse_s3_uri(prefix.rstrip('/') + '/')
    paginator=client.get_paginator('list_objects_v2'); batch=[]
    for page in paginator.paginate(Bucket=bucket, Prefix=key_prefix):
        for item in page.get('Contents', []):
            batch.append({'Key': item['Key']})
            if len(batch)==1000:
                client.delete_objects(Bucket=bucket, Delete={'Objects': batch}); batch=[]
    if batch: client.delete_objects(Bucket=bucket, Delete={'Objects': batch})

def upload_parquet_df(df: pd.DataFrame, uri: str, compression='zstd', client=None):
    import pyarrow as pa, pyarrow.parquet as pq
    client=client or boto3.client('s3'); table=pa.Table.from_pandas(df, preserve_index=False); buf=io.BytesIO()
    pq.write_table(table, buf, compression=compression); buf.seek(0); bucket,key=parse_s3_uri(uri); client.upload_fileobj(buf,bucket,key)

def write_json_s3(obj: Any, uri: str, client=None):
    client=client or boto3.client('s3'); bucket,key=parse_s3_uri(uri)
    client.put_object(Bucket=bucket, Key=key, Body=json.dumps(obj, indent=2, default=str).encode('utf-8'), ContentType='application/json')

def write_partitioned_df(df: pd.DataFrame, output_prefix: str, partition_col: str, file_tag: str, compression: str, client=None):
    if df.empty: return
    for value, group in df.groupby(partition_col, dropna=False):
        safe=str(value).replace('/','_'); uri=output_prefix.rstrip('/')+f'/{partition_col}={safe}/stage-{file_tag}-{uuid.uuid4().hex}.parquet'
        upload_parquet_df(group, uri, compression, client)

def list_partition_values(prefix: str, partition_col: str, client=None) -> list[str]:
    client=client or boto3.client('s3'); bucket,key_prefix=parse_s3_uri(prefix.rstrip('/')+'/'); paginator=client.get_paginator('list_objects_v2'); vals=set(); marker=f'{partition_col}='
    for page in paginator.paginate(Bucket=bucket, Prefix=key_prefix):
        for item in page.get('Contents', []):
            rel=item['Key'][len(key_prefix):]; part=rel.split('/',1)[0]
            if part.startswith(marker): vals.add(part[len(marker):])
    return sorted(vals)

def compact_partition(prefix: str, partition_col: str, value: str, compression='zstd', client=None):
    import pyarrow as pa, pyarrow.parquet as pq, s3fs
    client=client or boto3.client('s3'); fs=s3fs.S3FileSystem(anon=False)
    pp=prefix.rstrip('/')+f'/{partition_col}={value}/'; files=list_s3_files(pp,('.parquet',),client); stages=[f for f in files if '/stage-' in f]
    if len(stages)<=1: return
    with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
        writer=None
        try:
            for uri in stages:
                with fs.open(uri,'rb') as fh:
                    pf=pq.ParquetFile(fh)
                    for batch in pf.iter_batches(batch_size=100_000):
                        table=pa.Table.from_batches([batch])
                        if writer is None: writer=pq.ParquetWriter(tmp.name, table.schema, compression=compression)
                        writer.write_table(table)
            if writer:
                writer.close(); writer=None; final=pp+'compact.parquet'; bucket,key=parse_s3_uri(final); client.upload_file(tmp.name,bucket,key)
                for uri in stages:
                    b,k=parse_s3_uri(uri); client.delete_object(Bucket=b,Key=k)
        finally:
            if writer: writer.close()

def compact_all_partitions(prefix: str, partition_col: str, compression='zstd', client=None):
    client=client or boto3.client('s3'); vals=list_partition_values(prefix,partition_col,client)
    for i,v in enumerate(vals,1):
        logger.info('Compacting %s=%s (%s/%s)',partition_col,v,i,len(vals)); compact_partition(prefix,partition_col,v,compression,client)

def tile_name(ix:int, iy:int)->str: return f'E{ix:03d}_N{iy:03d}'

def geometry_tiles(geom, tile_size=10_000.0)->list[str]:
    if geom is None or geom.is_empty: return []
    minx,miny,maxx,maxy=geom.bounds; x0=int(math.floor(minx/tile_size)); x1=int(math.floor(maxx/tile_size)); y0=int(math.floor(miny/tile_size)); y1=int(math.floor(maxy/tile_size))
    return [tile_name(x,y) for x in range(x0,x1+1) for y in range(y0,y1+1)]

def tiles_for_bbox(bounds, tile_size=10_000.0)->list[str]:
    minx,miny,maxx,maxy=bounds; x0=int(math.floor(minx/tile_size)); x1=int(math.floor(maxx/tile_size)); y0=int(math.floor(miny/tile_size)); y1=int(math.floor(maxy/tile_size))
    return [tile_name(x,y) for x in range(x0,x1+1) for y in range(y0,y1+1)]

def write_geodataframe_tiles(gdf:gpd.GeoDataFrame, output_prefix:str, file_tag:str, compression='zstd'):
    if gdf.empty: return
    if gdf.crs is None: raise ValueError('Spatial source has no CRS; refusing to guess')
    if gdf.crs.to_epsg()!=27700: gdf=gdf.to_crs(27700)
    gdf=gdf.copy(); gdf['tile10km']=gdf.geometry.map(geometry_tiles); gdf=gdf.explode('tile10km',ignore_index=True)
    for tile,group in gdf.groupby('tile10km'):
        uri=output_prefix.rstrip('/')+f'/tile10km={tile}/stage-{file_tag}-{uuid.uuid4().hex}.parquet'; group.to_parquet(uri,compression=compression,index=False)

## Preprocess Source Data

In [ ]:
# --------------------------- PREPROCESS SOURCE DATA ---------------------------
def preprocess_addressbase(config: AppConfig) -> dict:
    client=s3_client_for(config); files=list_s3_files(config.raw.addressbase_core,('.csv',),client)
    if not files: raise FileNotFoundError(f'No AddressBase CSV under {config.raw.addressbase_core}')
    total=0
    for fi,uri in enumerate(files):
        logger.info('AddressBase %s',uri); sample=pd.read_csv(uri,nrows=5,low_memory=False); cmap=map_columns(sample.columns,ADDRESSBASE_REQUIRED,ADDRESSBASE_OPTIONAL)
        for ci,raw in enumerate(pd.read_csv(uri,chunksize=config.preprocess.csv_chunksize,low_memory=False)):
            def get(target, default=None):
                src=cmap.get(target)
                return raw[src] if src and src in raw.columns else pd.Series([default]*len(raw),index=raw.index)
            rec=pd.DataFrame({
                'uprn':get('uprn').astype('string'),'full_address':get('full_address').astype('string'),'postcode':get('postcode').astype('string'),'town':get('town').astype('string'),
                'classification_code':get('classification_code').astype('string'),'easting':pd.to_numeric(get('easting'),errors='coerce'),'northing':pd.to_numeric(get('northing'),errors='coerce'),
                'latitude':pd.to_numeric(get('latitude'),errors='coerce'),'longitude':pd.to_numeric(get('longitude'),errors='coerce'),'usrn':get('usrn').astype('string'),'toid':get('toid').astype('string')})
            rec=rec.dropna(subset=['uprn','full_address','easting','northing']).copy(); rec['uprn_bucket']=rec['uprn'].map(lambda x:stable_bucket(x,config.preprocess.uprn_bucket_count))
            tag=f'{fi:04d}-{ci:05d}'; write_partitioned_df(rec,config.processed.address_records,'uprn_bucket',tag,config.preprocess.parquet_compression,client)
            search=rec['full_address'].fillna('')+' '+rec['postcode'].fillna('')+' '+rec['town'].fillna(''); posts=build_token_rows_vectorized(rec['uprn'],search,'uprn',config.preprocess.token_prefix_chars)
            write_partitioned_df(posts,config.processed.address_tokens,'token_prefix',tag,config.preprocess.parquet_compression,client); total+=len(rec)
    if config.preprocess.compact_after_processing:
        compact_all_partitions(config.processed.address_records,'uprn_bucket',config.preprocess.parquet_compression,client); compact_all_partitions(config.processed.address_tokens,'token_prefix',config.preprocess.parquet_compression,client)
    return {'files':len(files),'rows':total}

def _read_open_names_header(files:list[str])->Optional[list[str]]:
    for uri in [f for f in files if 'header' in Path(f).name.lower()]:
        try:
            cols=list(pd.read_csv(uri,nrows=0).columns)
            if any(normalize_text(c).replace(' ','_') in {'name1','geometry_x','local_type'} for c in cols): return cols
        except Exception: pass
        raw=pd.read_csv(uri,header=None,nrows=2)
        if len(raw):
            row=[str(x).strip() for x in raw.iloc[0].tolist()]
            if any(normalize_text(c).replace(' ','_') in {'name1','geometry_x','local_type'} for c in row): return row
    return None

def _open_names_reader(uri:str, header_names:Optional[list[str]], chunksize:int):
    probe=pd.read_csv(uri,nrows=2,low_memory=False)
    try:
        map_columns(probe.columns,OPEN_NAMES_REQUIRED); return pd.read_csv(uri,chunksize=chunksize,low_memory=False)
    except Exception:
        if not header_names: raise ValueError(f'Open Names seems headerless and no header CSV found: {uri}')
        return pd.read_csv(uri,names=header_names,header=None,chunksize=chunksize,low_memory=False)

def preprocess_open_names(config:AppConfig)->dict:
    client=s3_client_for(config); all_files=list_s3_files(config.raw.open_names,('.csv',),client)
    if not all_files: raise FileNotFoundError(f'No Open Names CSV under {config.raw.open_names}')
    headers=_read_open_names_header(all_files); files=[f for f in all_files if 'header' not in Path(f).name.lower()]; total=0
    for fi,uri in enumerate(files):
        logger.info('Open Names %s',uri); reader=_open_names_reader(uri,headers,config.preprocess.csv_chunksize); cmap=None
        for ci,raw in enumerate(reader):
            if cmap is None: cmap=map_columns(raw.columns,OPEN_NAMES_REQUIRED)
            rec=pd.DataFrame({'name_id':raw[cmap['name_id']].astype('string'),'name':raw[cmap['name']].astype('string'),'feature_type':raw[cmap['feature_type']].astype('string'),
                              'easting':pd.to_numeric(raw[cmap['easting']],errors='coerce'),'northing':pd.to_numeric(raw[cmap['northing']],errors='coerce')})
            rec=rec.dropna(subset=['name_id','name','easting','northing']).copy(); rec['name_bucket']=rec['name_id'].map(lambda x:stable_bucket(x,config.preprocess.name_bucket_count)); tag=f'{fi:04d}-{ci:05d}'
            write_partitioned_df(rec,config.processed.name_records,'name_bucket',tag,config.preprocess.parquet_compression,client); posts=build_token_rows_vectorized(rec['name_id'],rec['name']+' '+rec['feature_type'],'name_id',config.preprocess.token_prefix_chars)
            write_partitioned_df(posts,config.processed.name_tokens,'token_prefix',tag,config.preprocess.parquet_compression,client); total+=len(rec)
    if config.preprocess.compact_after_processing:
        compact_all_partitions(config.processed.name_records,'name_bucket',config.preprocess.parquet_compression,client); compact_all_partitions(config.processed.name_tokens,'token_prefix',config.preprocess.parquet_compression,client)
    return {'files':len(files),'rows':total,'header_detected':bool(headers)}

def _canonical_uprn(raw:pd.DataFrame,cmap:dict)->pd.DataFrame:
    out=pd.DataFrame({'uprn':raw[cmap['uprn']].astype('string'),'easting':pd.to_numeric(raw[cmap['easting']],errors='coerce'),'northing':pd.to_numeric(raw[cmap['northing']],errors='coerce'),
                      'latitude':pd.to_numeric(raw[cmap['latitude']],errors='coerce'),'longitude':pd.to_numeric(raw[cmap['longitude']],errors='coerce')})
    return out.dropna(subset=['uprn','easting','northing']).copy()

def preprocess_open_uprn(config:AppConfig)->dict:
    client=s3_client_for(config); csvs=list_s3_files(config.raw.open_uprn,('.csv',),client); gpkgs=list_s3_files(config.raw.open_uprn,('.gpkg',),client); total=0; fi=0
    for uri in csvs:
        logger.info('Open UPRN %s',uri); probe=pd.read_csv(uri,nrows=5,low_memory=False)
        try: cmap=map_columns(probe.columns,OPEN_UPRN_REQUIRED); reader=pd.read_csv(uri,chunksize=config.preprocess.csv_chunksize,low_memory=False)
        except Exception:
            names=['UPRN','X_COORDINATE','Y_COORDINATE','LATITUDE','LONGITUDE']; cmap=map_columns(names,OPEN_UPRN_REQUIRED); reader=pd.read_csv(uri,names=names,header=None,chunksize=config.preprocess.csv_chunksize,low_memory=False)
        for ci,raw in enumerate(reader):
            rec=_canonical_uprn(raw,cmap); rec['uprn_bucket']=rec['uprn'].map(lambda x:stable_bucket(x,config.preprocess.uprn_bucket_count)); write_partitioned_df(rec,config.processed.uprn_records,'uprn_bucket',f'{fi:04d}-{ci:05d}',config.preprocess.parquet_compression,client); total+=len(rec)
        fi+=1
    for uri in gpkgs:
        bucket,key=parse_s3_uri(uri)
        with tempfile.NamedTemporaryFile(suffix='.gpkg') as tmp:
            client.download_file(bucket,key,tmp.name)
            for layer in fiona.listlayers(tmp.name):
                gdf=gpd.read_file(tmp.name,layer=layer); cmap=map_columns(gdf.columns,OPEN_UPRN_REQUIRED); rec=_canonical_uprn(gdf,cmap); rec['uprn_bucket']=rec['uprn'].map(lambda x:stable_bucket(x,config.preprocess.uprn_bucket_count)); write_partitioned_df(rec,config.processed.uprn_records,'uprn_bucket',f'{fi:04d}-{normalize_text(layer)}',config.preprocess.parquet_compression,client); total+=len(rec)
        fi+=1
    if not csvs and not gpkgs: raise FileNotFoundError(f'No Open UPRN CSV/GPKG under {config.raw.open_uprn}')
    if config.preprocess.compact_after_processing: compact_all_partitions(config.processed.uprn_records,'uprn_bucket',config.preprocess.parquet_compression,client)
    return {'files':len(csvs)+len(gpkgs),'rows':total}

def preprocess_open_roads(config:AppConfig)->dict:
    client=s3_client_for(config); files=list_s3_files(config.raw.open_roads,('.gpkg',),client)
    if not files: raise FileNotFoundError(f'No Open Roads GPKG under {config.raw.open_roads}')
    total=0
    for fi,uri in enumerate(files):
        logger.info('Open Roads %s',uri); bucket,key=parse_s3_uri(uri)
        with tempfile.NamedTemporaryFile(suffix='.gpkg') as tmp:
            client.download_file(bucket,key,tmp.name); layers=list(fiona.listlayers(tmp.name)); road_layers=[x for x in layers if 'roadlink' in normalize_text(x).replace(' ','')]
            if not road_layers: raise ValueError(f'No RoadLink layer in {uri}; layers={layers}')
            for layer in road_layers:
                gdf=gpd.read_file(tmp.name,layer=layer)
                if gdf.crs is None: raise ValueError(f'Open Roads layer has no CRS: {uri}/{layer}')
                gdf=gdf.to_crs(27700); idc=find_column(gdf.columns,['id','identifier','gml_id','fid'],False); n1=find_column(gdf.columns,['name_1','name1','name'],False); n2=find_column(gdf.columns,['name_2','name2'],False); rn=find_column(gdf.columns,['road_classification_number','roadClassificationNumber','road_number'],False)
                out=gpd.GeoDataFrame({'feature_id':gdf[idc].astype('string') if idc else pd.Series([f'road-{uuid.uuid4().hex}' for _ in range(len(gdf))]),
                                      'name':gdf[n1].astype('string') if n1 else pd.Series(['']*len(gdf)), 'name2':gdf[n2].astype('string') if n2 else pd.Series(['']*len(gdf)),
                                      'road_number':gdf[rn].astype('string') if rn else pd.Series(['']*len(gdf)), 'geometry':gdf.geometry},geometry='geometry',crs=27700)
                out=out[out.geometry.notna() & (~out.geometry.is_empty)].copy(); write_geodataframe_tiles(out,config.processed.road_geometries,f'{fi:04d}-{normalize_text(layer)}',config.preprocess.parquet_compression); total+=len(out)
    if config.preprocess.compact_after_processing: compact_all_partitions(config.processed.road_geometries,'tile10km',config.preprocess.parquet_compression,client)
    return {'files':len(files),'rows':total}

def _find_inspire_id_column(columns)->str:
    return find_column(columns,['inspire_id','INSPIRE_ID','inspireid','localId','local_id','identifier'],True)

def preprocess_hmlr_inspire(config:AppConfig)->dict:
    client=s3_client_for(config); files=list_s3_files(config.raw.hmlr_inspire,('.gml',),client)
    if not files: raise FileNotFoundError(f'No HMLR GML under {config.raw.hmlr_inspire}')
    total=0
    for fi,uri in enumerate(files):
        logger.info('HMLR INSPIRE %s/%s %s',fi+1,len(files),uri); bucket,key=parse_s3_uri(uri)
        with tempfile.NamedTemporaryFile(suffix='.gml') as tmp:
            client.download_file(bucket,key,tmp.name); gdf=gpd.read_file(tmp.name)
            if gdf.crs is None: raise ValueError(f'HMLR file has no CRS: {uri}')
            gdf=gdf.to_crs(27700); idc=_find_inspire_id_column(gdf.columns)
            out=gpd.GeoDataFrame({'inspire_id':gdf[idc].astype('string'),'geometry':gdf.geometry},geometry='geometry',crs=27700); out['geometry']=out.geometry.map(lambda g:make_valid(g) if g is not None else g); out=out[out.geometry.notna() & (~out.geometry.is_empty)].copy()
            b=out.geometry.bounds; out['minx']=b.minx; out['miny']=b.miny; out['maxx']=b.maxx; out['maxy']=b.maxy; out['source_file']=Path(uri).name
            write_geodataframe_tiles(out,config.processed.inspire,f'{fi:05d}',config.preprocess.parquet_compression); total+=len(out)
    if config.preprocess.compact_after_processing: compact_all_partitions(config.processed.inspire,'tile10km',config.preprocess.parquet_compression,client)
    return {'files':len(files),'rows':total}

def inspect_raw_data(config:AppConfig)->dict:
    client=s3_client_for(config); report={
        'addressbase_files':list_s3_files(config.raw.addressbase_core,('.csv',),client),
        'open_names_files':list_s3_files(config.raw.open_names,('.csv',),client),
        'open_uprn_files':list_s3_files(config.raw.open_uprn,('.csv','.gpkg'),client),
        'open_roads_files':list_s3_files(config.raw.open_roads,('.gpkg',),client),
        'hmlr_files':list_s3_files(config.raw.hmlr_inspire,('.gml',),client),}
    if report['addressbase_files']:
        sample=pd.read_csv(report['addressbase_files'][0],nrows=3,low_memory=False); report['addressbase_columns']=list(sample.columns); report['addressbase_mapping']=map_columns(sample.columns,ADDRESSBASE_REQUIRED,ADDRESSBASE_OPTIONAL)
    if report['open_names_files']: report['open_names_header_detected']=bool(_read_open_names_header(report['open_names_files']))
    return report

def run_full_preprocessing(config:AppConfig, overwrite=False)->dict:
    client=s3_client_for(config); outputs=[config.processed.address_records,config.processed.address_tokens,config.processed.name_records,config.processed.name_tokens,config.processed.uprn_records,config.processed.road_geometries,config.processed.inspire]
    if overwrite:
        for p in outputs: logger.warning('Deleting processed prefix %s',p); delete_s3_prefix(p,client)
    report={'addressbase':preprocess_addressbase(config),'open_names':preprocess_open_names(config),'open_uprn':preprocess_open_uprn(config),'open_roads':preprocess_open_roads(config),'hmlr_inspire':preprocess_hmlr_inspire(config)}
    write_json_s3(report,config.processed.results.rstrip('/')+'/_preprocessing_report.json',client); return report

## Llm Structures

In [ ]:
# --------------------------- LLM STRUCTURES ---------------------------
AnchorType=Literal['address','property','road','landmark','settlement','watercourse','railway','other','unknown']
RelationType=Literal['at','adjacent_to','between','rear_of','front_of','opposite','north_of','south_of','east_of','west_of','on_side_of','bounded_by','near','accessed_from','part_of','at_junction_of','other']
StrengthType=Literal['required','strong','medium','weak']

class AnchorSpec(BaseModel):
    model_config=ConfigDict(extra='forbid')
    id:str; type:AnchorType; raw_text:str; canonical_query:str; search_variants:list[str]
    postcode:Optional[str]=None; town_or_locality:Optional[str]=None; may_represent_multiple_properties:bool=False; extraction_confidence:float=Field(ge=0,le=1)

class SpatialConstraint(BaseModel):
    model_config=ConfigDict(extra='forbid')
    relation:RelationType; anchor_ids:list[str]; direction:Optional[str]=None; strength:StrengthType='medium'; original_phrase:str

class ParsedDescription(BaseModel):
    model_config=ConfigDict(extra='forbid')
    subject_type:Literal['land','building','property','garage','unit','other','unknown']; subject_name:Optional[str]=None; locality_context:Optional[str]=None; postcode_context:Optional[str]=None
    anchors:list[AnchorSpec]; constraints:list[SpatialConstraint]; normalized_description:str; ambiguity_notes:list[str]

class ParsedAudit(BaseModel):
    model_config=ConfigDict(extra='forbid')
    is_complete:bool; corrected:ParsedDescription; corrections_made:list[str]

class SemanticCandidateAssessment(BaseModel):
    model_config=ConfigDict(extra='forbid')
    inspire_id:str; consistency:float=Field(ge=0,le=1); serious_contradiction:bool; explanation:str

class SemanticAssessment(BaseModel):
    model_config=ConfigDict(extra='forbid')
    candidates:list[SemanticCandidateAssessment]; notes:list[str]

DESCRIPTION_PROMPT='''
You parse complex land/property descriptions for England and Wales. You are NOT a geocoder.
Never invent UPRNs, coordinates, title numbers, postcodes, towns, roads or landmarks.
Distinguish the unknown subject land/property from every reference anchor.
Examples:
- "land adjacent to 12 High Street": 12 High Street is an anchor, adjacent_to.
- "land between 10 and 14 Station Road": create separate usable references to 10 and 14 and one between constraint referencing both.
- "land adjacent to 12 and 14 High Street": create separate anchors for 12 High Street and 14 High Street when the wording clearly refers to two explicit properties.
- "land to the rear of 10-16 High Street": preserve the range, mark it potentially multiple, rear_of.
- "bounded on the east by Mill Lane": Mill Lane is a road anchor, bounded_by, direction east.
- "on the north side of Station Road": on_side_of, direction north.
- "at the junction of Mill Lane and Church Road": both road anchors, at_junction_of.
Search variants may expand safe abbreviations/inherited road/locality context but must not invent new geographic facts. Max 5 variants per anchor.
Preserve uncertainty in ambiguity_notes rather than guessing.
'''
AUDIT_PROMPT='''
Audit a structured parse against the ORIGINAL DESCRIPTION. Correct missed anchors, wrong subject/reference distinction, missed/wrong spatial relations, inherited street/locality context, and grouping of between/rear/bounded/opposite/directions/junctions. Never add geographic facts not in the original. Return the complete corrected ParsedDescription.
'''
SEMANTIC_PROMPT='''
Assess candidate HMLR INSPIRE polygons against the original England/Wales land description using ONLY supplied parsed text and deterministic GIS evidence. OS/HMLR/GIS evidence is geographic truth. Never invent geographic facts. Missing evidence is unverifiable, not contradictory. Real contradictions with required clues matter. INSPIRE extents are indicative, so do not demand survey precision.
'''

class DescriptionParser:
    def __init__(self,client:OpenAI,config:AppConfig): self.client=client; self.config=config
    def parse_once(self,text:str)->ParsedDescription:
        r=self.client.responses.parse(model=self.config.azure_deployment,instructions=DESCRIPTION_PROMPT,input=text,text_format=ParsedDescription); return r.output_parsed
    def parse(self,text:str)->ParsedDescription:
        if not text or not text.strip(): raise ValueError('Description is empty')
        first=self.parse_once(text)
        if not self.config.llm.parse_twice: return first
        r=self.client.responses.parse(model=self.config.azure_deployment,instructions=AUDIT_PROMPT,input=json.dumps({'original_description':text,'parsed_structure':first.model_dump()}),text_format=ParsedAudit)
        return r.output_parsed.corrected

## Processed Data Readers

In [ ]:
# --------------------------- PROCESSED DATA READERS ---------------------------
class PartitionedParquetReader:
    def __init__(self,prefix:str,config:AppConfig): self.prefix=prefix.rstrip('/'); self.config=config; self.client=s3_client_for(config)
    def files(self,partition_col:str,value:str)->list[str]: return list_s3_files(self.prefix+f'/{partition_col}={value}/',('.parquet',),self.client)
    def read_df(self,partition_col:str,value:str,columns:Optional[list[str]]=None)->pd.DataFrame:
        frames=[]
        for uri in self.files(partition_col,value):
            try: frames.append(pd.read_parquet(uri,columns=columns))
            except Exception as e: logger.warning('Parquet read failed %s: %s',uri,e)
        return pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=columns or [])

class TokenPostingIndex:
    def __init__(self,prefix:str,id_col:str,config:AppConfig): self.reader=PartitionedParquetReader(prefix,config); self.id_col=id_col; self.config=config; self._cache={}
    def postings(self,token:str)->pd.Series:
        token=normalize_text(token)
        if not token: return pd.Series(dtype='string')
        pref=token_prefix(token,self.config.preprocess.token_prefix_chars)
        if pref not in self._cache: self._cache[pref]=self.reader.read_df('token_prefix',pref,[self.id_col,'token'])
        df=self._cache[pref]
        if df.empty:return pd.Series(dtype='string')
        return df.loc[df['token']==token,self.id_col].astype('string').drop_duplicates()

class OpenUPRNRepository:
    def __init__(self,config:AppConfig): self.config=config; self.reader=PartitionedParquetReader(config.processed.uprn_records,config)
    def get_many(self,uprns:Iterable[str])->dict[str,dict]:
        wanted=list(dict.fromkeys(str(x) for x in uprns if x)); groups={}; out={}
        for u in wanted: groups.setdefault(stable_bucket(u,self.config.preprocess.uprn_bucket_count),[]).append(u)
        for b,vals in groups.items():
            df=self.reader.read_df('uprn_bucket',str(b));
            if df.empty: continue
            sub=df[df['uprn'].astype('string').isin(vals)].drop_duplicates('uprn')
            for _,r in sub.iterrows(): out[str(r['uprn'])]=r.to_dict()
        return out

class AddressRepository:
    def __init__(self,config:AppConfig,uprn_repo:OpenUPRNRepository):
        self.config=config; self.uprn_repo=uprn_repo; self.records=PartitionedParquetReader(config.processed.address_records,config); self.index=TokenPostingIndex(config.processed.address_tokens,'uprn',config)
    def get_many(self,uprns:Iterable[str])->pd.DataFrame:
        wanted=list(dict.fromkeys(str(x) for x in uprns if x)); groups={}; frames=[]
        for u in wanted: groups.setdefault(stable_bucket(u,self.config.preprocess.uprn_bucket_count),[]).append(u)
        for b,vals in groups.items():
            df=self.records.read_df('uprn_bucket',str(b));
            if not df.empty: frames.append(df[df['uprn'].astype('string').isin(vals)])
        return pd.concat(frames,ignore_index=True).drop_duplicates('uprn') if frames else pd.DataFrame()
    def _candidate_ids(self,tokens:list[str])->set[str]:
        sets=[]
        for t in tokens[:self.config.retrieval.max_query_tokens]:
            ids=self.index.postings(t); n=len(ids)
            if 0<n<=self.config.retrieval.max_postings_per_token: sets.append((t,set(ids.astype(str))))
        if not sets:return set()
        sets.sort(key=lambda x:len(x[1])); pool=set(sets[0][1])
        for _,s in sets[1:]:
            inter=pool & s
            if inter: pool=inter
            elif len(pool)<20: pool |= s
            if len(pool)<=100: break
        if len(pool)>self.config.retrieval.max_address_candidate_pool: pool=set(sorted(pool)[:self.config.retrieval.max_address_candidate_pool])
        return pool
    def search(self,query:str,postcode:Optional[str]=None,town:Optional[str]=None,limit:Optional[int]=None)->list[dict]:
        limit=limit or self.config.retrieval.address_top_k; combined=' '.join(x for x in [query,postcode or '',town or ''] if x); ids=self._candidate_ids(tokenize(combined))
        if not ids:
            for v in address_query_variants(query):
                ids=self._candidate_ids(tokenize(' '.join(x for x in [v,postcode or '',town or ''] if x)))
                if ids: break
        if not ids:return []
        df=self.get_many(ids)
        if df.empty:return []
        q=normalize_text(query); pc=normalize_text(postcode).replace(' ',''); tn=normalize_text(town)
        def score(r):
            full=normalize_text(r.get('full_address')); base=fuzz.WRatio(q,full); qt=set(tokenize(query)); ft=set(tokenize(full)); coverage=len(qt&ft)/max(1,len(qt)); s=.75*base+25*coverage
            if pc:
                rpc=normalize_text(r.get('postcode')).replace(' ','')
                if rpc==pc:s+=10
                elif rpc and len(pc)>=3 and rpc[:3]==pc[:3]:s+=4
            if tn and tn in normalize_text(r.get('town')):s+=6
            return min(100.,float(s))
        df['_match_score']=df.apply(score,axis=1); df=df[df['_match_score']>=self.config.retrieval.min_address_score].sort_values('_match_score',ascending=False).head(limit)
        validation=self.uprn_repo.get_many(df['uprn'].astype(str).tolist()); out=[]
        for _,r in df.iterrows():
            u=str(r['uprn']); x=float(r['easting']); y=float(r['northing']); v=validation.get(u); source='ADDRESSBASE'; disagreement=None; penalty=1.
            if v:
                ux=float(v['easting']); uy=float(v['northing']); disagreement=math.hypot(x-ux,y-uy)
                if disagreement>self.config.retrieval.uprn_coordinate_hard_disagreement_m: penalty=.65
                elif disagreement>self.config.retrieval.uprn_coordinate_warning_m: penalty=.85
                x,y=ux,uy; source='OS_OPEN_UPRN'
            out.append({'source':'OS_ADDRESSBASE','source_id':u,'uprns':[u],'label':str(r['full_address']),'member_labels':[str(r['full_address'])],
                        'easting':x,'northing':y,'match_score':float(r['_match_score'])*penalty,'postcode':None if pd.isna(r.get('postcode')) else str(r.get('postcode')),
                        'town':None if pd.isna(r.get('town')) else str(r.get('town')),'coordinate_source':source,'coordinate_disagreement_m':disagreement})
        return out

class NameRepository:
    def __init__(self,config:AppConfig): self.config=config; self.records=PartitionedParquetReader(config.processed.name_records,config); self.index=TokenPostingIndex(config.processed.name_tokens,'name_id',config)
    def get_many(self,ids:Iterable[str])->pd.DataFrame:
        wanted=list(dict.fromkeys(str(x) for x in ids if x)); groups={}; frames=[]
        for n in wanted: groups.setdefault(stable_bucket(n,self.config.preprocess.name_bucket_count),[]).append(n)
        for b,vals in groups.items():
            df=self.records.read_df('name_bucket',str(b));
            if not df.empty: frames.append(df[df['name_id'].astype('string').isin(vals)])
        return pd.concat(frames,ignore_index=True).drop_duplicates('name_id') if frames else pd.DataFrame()
    def search(self,query:str,feature_type_hint:Optional[str]=None,limit:Optional[int]=None)->list[dict]:
        limit=limit or self.config.retrieval.name_top_k; sets=[]
        for t in tokenize(query)[:self.config.retrieval.max_query_tokens]:
            ids=self.index.postings(t)
            if 0<len(ids)<=self.config.retrieval.max_postings_per_token: sets.append(set(ids.astype(str)))
        if not sets:return []
        sets.sort(key=len); pool=set(sets[0])
        for s in sets[1:]:
            inter=pool&s
            if inter:pool=inter
        if len(pool)>self.config.retrieval.max_name_candidate_pool: pool=set(sorted(pool)[:self.config.retrieval.max_name_candidate_pool])
        df=self.get_many(pool)
        if df.empty:return []
        q=normalize_text(query); hint=normalize_text(feature_type_hint)
        def score(r):
            s=fuzz.WRatio(q,normalize_text(r.get('name')))
            if hint and hint in normalize_text(r.get('feature_type')): s+=8
            return min(100.,float(s))
        df['_match_score']=df.apply(score,axis=1); df=df[df['_match_score']>=self.config.retrieval.min_name_score].sort_values('_match_score',ascending=False).head(limit)
        return [{'source':'OS_OPEN_NAMES','source_id':str(r['name_id']),'uprns':[],'label':str(r['name']),'member_labels':[str(r['name'])],'feature_type':str(r.get('feature_type') or ''),'easting':float(r['easting']),'northing':float(r['northing']),'match_score':float(r['_match_score'])} for _,r in df.iterrows()]

## Spatial Repositories

In [ ]:
# --------------------------- SPATIAL REPOSITORIES ---------------------------
class TiledGeoRepository:
    def __init__(self,prefix:str,config:AppConfig): self.prefix=prefix.rstrip('/'); self.config=config; self.client=s3_client_for(config)
    def read_tiles(self,tiles:list[str])->gpd.GeoDataFrame:
        frames=[]
        for tile in tiles:
            for uri in list_s3_files(self.prefix+f'/tile10km={tile}/',('.parquet',),self.client):
                try: frames.append(gpd.read_parquet(uri))
                except Exception as e: logger.warning('GeoParquet read failed %s: %s',uri,e)
        if not frames:return gpd.GeoDataFrame(geometry=[],crs=27700)
        g=gpd.GeoDataFrame(pd.concat(frames,ignore_index=True),geometry='geometry')
        if g.crs is None:g=g.set_crs(27700)
        elif g.crs.to_epsg()!=27700:g=g.to_crs(27700)
        return g

class RoadRepository:
    def __init__(self,config:AppConfig): self.config=config; self.repo=TiledGeoRepository(config.processed.road_geometries,config)
    def in_bbox(self,bounds)->gpd.GeoDataFrame:
        g=self.repo.read_tiles(tiles_for_bbox(bounds))
        if g.empty:return g
        g=g[g.intersects(box(*bounds))].copy()
        if 'feature_id' in g.columns:g=g.drop_duplicates('feature_id')
        return g
    def match_named_road(self,name:str,around:Point,radius_m=1000.,limit=5)->list[dict]:
        g=self.in_bbox((around.x-radius_m,around.y-radius_m,around.x+radius_m,around.y+radius_m))
        if g.empty:return []
        q=normalize_text(name)
        def display(r):return ' '.join(str(r.get(k) or '') for k in ['name','name2','road_number']).strip()
        g['_name_score']=g.apply(lambda r:fuzz.WRatio(q,normalize_text(display(r))),axis=1); g['_distance']=g.geometry.distance(around); g['_score']=.85*g['_name_score']+.15*np.maximum(0,100-g['_distance']/10.)
        g=g.sort_values('_score',ascending=False).head(limit)
        return [{'feature_id':str(r.get('feature_id')),'label':display(r) or name,'match_score':float(r['_score']),'geometry':r.geometry} for _,r in g.iterrows()]
    def nearest_road(self,point:Point,radius_m=150.):
        g=self.in_bbox((point.x-radius_m,point.y-radius_m,point.x+radius_m,point.y+radius_m))
        if g.empty:return None
        d=g.geometry.distance(point); return g.loc[d.idxmin()].geometry

class InspireRepository:
    def __init__(self,config:AppConfig): self.config=config; self.repo=TiledGeoRepository(config.processed.inspire,config)
    def in_bbox(self,bounds,limit:Optional[int]=None)->gpd.GeoDataFrame:
        limit=limit or self.config.retrieval.hmlr_candidate_limit_per_hypothesis; query=box(*bounds); g=self.repo.read_tiles(tiles_for_bbox(bounds))
        if g.empty:return g
        g=g[g.intersects(query)].copy()
        if 'inspire_id' in g.columns:
            g['_geom_hash']=g.geometry.map(lambda x:hashlib.blake2b(x.wkb,digest_size=8).hexdigest()); g=g.drop_duplicates(['inspire_id','_geom_hash']).drop(columns='_geom_hash')
        if len(g)>limit:
            c=query.centroid; g['_d']=g.geometry.distance(c); g=g.sort_values('_d').head(limit).drop(columns='_d')
        return g

## Anchor Resolution

In [ ]:
# --------------------------- ANCHOR RESOLUTION ---------------------------
@dataclass
class AnchorCandidate:
    candidate_id:str; anchor_id:str; source:str; label:str; point:Point; match_score:float
    uprns:list[str]=field(default_factory=list); member_labels:list[str]=field(default_factory=list); geometry:Any=None; metadata:dict=field(default_factory=dict)

@dataclass
class ResolvedAnchorSet:
    anchor:AnchorSpec; candidates:list[AnchorCandidate]

def cluster_address_results(anchor_id:str,results:list[dict],cluster_distance_m=90.,group_nearby=True)->list[AnchorCandidate]:
    if not results:return []
    results=sorted(results,key=lambda r:r['match_score'],reverse=True)
    if not group_nearby:
        return [AnchorCandidate(f'{anchor_id}-addr-{i}',anchor_id,'OS_ADDRESSBASE',r['label'],Point(float(r['easting']),float(r['northing'])),float(r['match_score']),list(r.get('uprns',[])),list(r.get('member_labels',[])),None,{'members':[r]}) for i,r in enumerate(results)]
    clusters=[]
    for r in results:
        p=Point(float(r['easting']),float(r['northing'])); placed=False
        for cluster in clusters:
            c=MultiPoint([Point(float(x['easting']),float(x['northing'])) for x in cluster]).centroid
            if p.distance(c)<=cluster_distance_m: cluster.append(r); placed=True; break
        if not placed:clusters.append([r])
    out=[]
    for i,cluster in enumerate(clusters):
        pts=[Point(float(r['easting']),float(r['northing'])) for r in cluster]; c=MultiPoint(pts).centroid; score=min(100.,float(np.mean([r['match_score'] for r in cluster]))+min(8.,max(0,len(cluster)-1)*2.))
        out.append(AnchorCandidate(f'{anchor_id}-addr-{i}',anchor_id,'OS_ADDRESSBASE',' | '.join(list(dict.fromkeys(r['label'] for r in cluster))[:5]),c,score,
                                   list(dict.fromkeys(u for r in cluster for u in r.get('uprns',[]))),list(dict.fromkeys(l for r in cluster for l in r.get('member_labels',[]))),None,{'members':cluster}))
    return sorted(out,key=lambda c:c.match_score,reverse=True)

class AnchorResolver:
    def __init__(self,config:AppConfig,address_repo:AddressRepository,name_repo:NameRepository,road_repo:RoadRepository): self.config=config; self.address_repo=address_repo; self.name_repo=name_repo; self.road_repo=road_repo
    def resolve_anchor(self,anchor:AnchorSpec,parsed:ParsedDescription)->ResolvedAnchorSet:
        postcode=anchor.postcode or parsed.postcode_context; town=anchor.town_or_locality or parsed.locality_context; terms=list(dict.fromkeys([anchor.canonical_query]+anchor.search_variants[:self.config.llm.max_search_variants_per_anchor]))
        if anchor.type in ('address','property'):
            raw=[]
            for term in terms:
                for v in address_query_variants(term): raw.extend(self.address_repo.search(v,postcode,town,self.config.retrieval.address_top_k))
            dedup={}
            for r in raw:
                k=(tuple(r.get('uprns',[])),r['label'])
                if k not in dedup or r['match_score']>dedup[k]['match_score']:dedup[k]=r
            group_nearby = anchor.may_represent_multiple_properties or any(len(address_query_variants(term)) > 1 for term in terms)
            candidates=cluster_address_results(anchor.id,list(dedup.values()),group_nearby=group_nearby)
        else:
            raw=[]
            for term in terms:raw.extend(self.name_repo.search(term,anchor.type,self.config.retrieval.name_top_k))
            dedup={}
            for r in raw:
                k=r['source_id']
                if k not in dedup or r['match_score']>dedup[k]['match_score']:dedup[k]=r
            candidates=[]
            for i,r in enumerate(sorted(dedup.values(),key=lambda x:x['match_score'],reverse=True)[:self.config.retrieval.max_candidates_per_anchor_for_beam]):
                p=Point(float(r['easting']),float(r['northing'])); geom=None
                if anchor.type=='road':
                    matches=self.road_repo.match_named_road(anchor.canonical_query,p,1200.,1)
                    if matches:geom=matches[0]['geometry']
                candidates.append(AnchorCandidate(f'{anchor.id}-name-{i}',anchor.id,r['source'],r['label'],p,float(r['match_score']),[],r.get('member_labels',[]),geom,r))
        candidates = candidates[:self.config.retrieval.max_candidates_per_anchor_for_beam]
        for c in candidates:
            c.metadata.setdefault('anchor_type', anchor.type)
        return ResolvedAnchorSet(anchor,candidates)
    def resolve_all(self,parsed:ParsedDescription)->list[ResolvedAnchorSet]:return [self.resolve_anchor(a,parsed) for a in parsed.anchors]

## Joint Hypotheses

In [ ]:
# --------------------------- JOINT HYPOTHESES ---------------------------
@dataclass
class GeoHypothesis:
    assignments:dict[str,AnchorCandidate]; lexical_score:float; coherence_score:float; score:float

class HypothesisBuilder:
    def __init__(self,config:AppConfig):self.config=config
    def _coherence(self,assignments)->float:
        pts=[c.point for c in assignments.values()]
        if len(pts)<=1:return 1.
        ds=[pts[i].distance(pts[j]) for i in range(len(pts)) for j in range(i+1,len(pts))]; maxd=max(ds); hard=.35 if maxd>self.config.retrieval.hard_anchor_separation_m else 1.; smooth=math.exp(-float(np.mean(ds))/max(1.,self.config.retrieval.hypothesis_distance_scale_m)); return float(hard*smooth)
    def build(self,resolved:list[ResolvedAnchorSet])->list[GeoHypothesis]:
        usable=[r for r in resolved if r.candidates]
        if not usable:return []
        usable.sort(key=lambda r:(len(r.candidates),-r.candidates[0].match_score)); beam=[GeoHypothesis({},0.,1.,0.)]
        for rs in usable:
            expanded=[]
            for state in beam:
                for cand in rs.candidates:
                    ass=dict(state.assignments); ass[rs.anchor.id]=cand; lex=float(np.mean([c.match_score/100. for c in ass.values()])); coh=self._coherence(ass); score=.72*lex+.28*coh; expanded.append(GeoHypothesis(ass,lex,coh,score))
            expanded.sort(key=lambda h:h.score,reverse=True); unique=[]; seen=set()
            for h in expanded:
                sig=tuple(sorted((aid,round(c.point.x/25),round(c.point.y/25)) for aid,c in h.assignments.items()))
                if sig in seen:continue
                seen.add(sig);unique.append(h)
                if len(unique)>=self.config.retrieval.beam_width:break
            beam=unique
        return beam

def hypothesis_search_bbox(h:GeoHypothesis,parsed:ParsedDescription,config:AppConfig):
    pts=[c.point for c in h.assignments.values()]
    if not pts:raise ValueError('Hypothesis has no points')
    if len(pts)==1:
        c=next(iter(h.assignments.values())); typ=next((a.type for a in parsed.anchors if a.id==c.anchor_id),'other')
        radius=config.retrieval.single_property_radius_m if typ in ('address','property') else config.retrieval.single_road_radius_m if typ=='road' else config.retrieval.single_settlement_radius_m if typ=='settlement' else config.retrieval.default_search_radius_m
        p=pts[0];return (p.x-radius,p.y-radius,p.x+radius,p.y+radius)
    mp=MultiPoint(pts); minx,miny,maxx,maxy=mp.bounds; spread=math.hypot(maxx-minx,maxy-miny); radius=min(max(config.retrieval.default_search_radius_m,spread*.8),config.retrieval.max_search_radius_m); return (minx-radius,miny-radius,maxx+radius,maxy+radius)

## Spatial Scoring

In [ ]:
# --------------------------- SPATIAL SCORING ---------------------------
def bearing_deg(dx:float,dy:float)->float:return (math.degrees(math.atan2(dx,dy))+360.)%360.
def angular_difference(a:float,b:float)->float:return abs((a-b+180.)%360.-180.)
def exp_distance_score(distance_m:float,scale_m:float)->float:return float(math.exp(-max(0.,distance_m)/max(1.,scale_m)))
def constraint_strength_multiplier(s:str)->float:return {'required':1.35,'strong':1.15,'medium':1.,'weak':.65}.get(s,1.)

class SpatialScorer:
    def __init__(self,config:AppConfig,road_repo:RoadRepository):self.config=config;self.road_repo=road_repo
    def _anchor_geometries(self,c:AnchorCandidate,local:gpd.GeoDataFrame)->list:
        geoms=[]
        if c.geometry is not None:geoms.append(c.geometry)
        if c.uprns and not local.empty:
            pts=[Point(float(m['easting']),float(m['northing'])) for m in c.metadata.get('members',[]) if 'easting' in m and 'northing' in m] or [c.point]
            for p in pts:
                d=local.geometry.distance(p); sub=local[d<=self.config.spatial.anchor_polygon_lookup_tolerance_m]
                geoms.extend(list(sub.geometry))
        if not geoms:geoms=[c.point]
        out=[];seen=set()
        for g in geoms:
            k=hashlib.blake2b(g.wkb,digest_size=8).hexdigest()
            if k not in seen:seen.add(k);out.append(g)
        return out
    def _anchors(self,constraint,h):return [h.assignments[x] for x in constraint.anchor_ids if x in h.assignments]
    @staticmethod
    def _same_extent(subject,geoms):return any(g.geom_type in ('Polygon','MultiPolygon') and subject.equals(g) for g in geoms)
    def evaluate(self,subject,constraint:SpatialConstraint,h:GeoHypothesis,local:gpd.GeoDataFrame)->dict:
        anchors=self._anchors(constraint,h)
        if not anchors:return {'evaluated':False,'score':0.,'reason':'anchor unresolved'}
        grouped=[self._anchor_geometries(a,local) for a in anchors]; flat=[g for gs in grouped for g in gs]; rel=constraint.relation
        if rel=='at':
            ds=[subject.distance(g) for g in flat]; d=min(ds); score=exp_distance_score(d,5.)
            if any(subject.contains(g) if g.geom_type=='Point' else subject.intersects(g) for g in flat):score=1.
            return {'evaluated':True,'score':score,'distance_m':float(d)}
        if rel=='adjacent_to':
            vals=[]
            for gs in grouped:
                if self._same_extent(subject,gs):vals.append(.03);continue
                d=min(subject.distance(g) for g in gs);vals.append(exp_distance_score(d,self.config.spatial.adjacent_tolerance_m))
            return {'evaluated':True,'score':float(np.mean(vals)),'per_anchor_scores':vals}
        if rel=='near':
            d=min(subject.distance(g) for g in flat);return {'evaluated':True,'score':exp_distance_score(d,self.config.spatial.near_distance_m),'distance_m':float(d)}
        if rel=='between':
            if len(anchors)<2:return {'evaluated':False,'score':0.,'reason':'between needs >=2 resolved anchors'}
            refs=[unary_union(gs).centroid for gs in grouped]; corridor=LineString(refs).buffer(self.config.spatial.between_corridor_m) if len(refs)==2 else MultiPoint(refs).convex_hull.buffer(self.config.spatial.between_corridor_m)
            overlap=subject.intersection(corridor).area/subject.area if subject.area>0 else 0.; centroid_in=corridor.contains(subject.centroid); score=min(1.,.72*overlap+.28*float(centroid_in))
            return {'evaluated':True,'score':float(score),'corridor_overlap_ratio':float(overlap),'centroid_in_corridor':bool(centroid_in)}
        bearings={'north_of':0.,'east_of':90.,'south_of':180.,'west_of':270.}
        if rel in bearings:
            c=subject.centroid; vals=[]; bs=[]
            for gs in grouped:
                p=unary_union(gs).centroid;b=bearing_deg(c.x-p.x,c.y-p.y);diff=angular_difference(b,bearings[rel]);vals.append(max(0.,1.-diff/self.config.spatial.directional_tolerance_deg));bs.append(b)
            return {'evaluated':True,'score':float(np.mean(vals)),'bearings':bs}
        if rel in ('rear_of','front_of'):
            vals=[];ev=[]
            for a,gs in zip(anchors,grouped):
                road=self.road_repo.nearest_road(a.point,180.)
                if road is None:continue
                nr=nearest_points(a.point,road)[1];vx=a.point.x-nr.x;vy=a.point.y-nr.y;norm=math.hypot(vx,vy)
                if norm<.5:continue
                ux,uy=vx/norm,vy/norm;c=subject.centroid;projection=(c.x-a.point.x)*ux+(c.y-a.point.y)*uy;side=projection>0 if rel=='rear_of' else projection<0;d=min(subject.distance(g) for g in gs);score=.65*float(side)+.35*exp_distance_score(d,self.config.spatial.rear_front_max_distance_m);vals.append(score);ev.append({'projection_m':float(projection),'distance_m':float(d)})
            return {'evaluated':bool(vals),'score':float(np.mean(vals)) if vals else 0.,'evidence':ev,'reason':None if vals else 'no nearby road to infer front/rear'}
        if rel=='opposite':
            vals=[];ev=[]
            for a,gs in zip(anchors,grouped):
                road=self.road_repo.nearest_road(a.point,180.)
                if road is None:continue
                connector=LineString([a.point,subject.centroid]);between=connector.intersects(road);d=min(subject.distance(g) for g in gs);score=.72*float(between)+.28*exp_distance_score(d,self.config.spatial.opposite_max_distance_m);vals.append(score);ev.append({'road_between':bool(between),'distance_m':float(d)})
            return {'evaluated':bool(vals),'score':float(np.mean(vals)) if vals else 0.,'evidence':ev,'reason':None if vals else 'no nearby road for opposite test'}
        if rel in ('bounded_by','on_side_of','accessed_from'):
            lookup={'north':0.,'east':90.,'south':180.,'west':270.}; desired=lookup.get(normalize_text(constraint.direction));vals=[];ev=[]
            for a,gs in zip(anchors,grouped):
                # A named point alone is not enough to prove a boundary/side relation.
                # Roads have line geometry; address/property anchors can have INSPIRE extents.
                if rel in ('bounded_by','on_side_of') and a.geometry is None and not a.uprns:
                    continue
                g=a.geometry if a.geometry is not None else unary_union(gs);d=subject.boundary.distance(g);scale=self.config.spatial.boundary_tolerance_m if rel!='accessed_from' else self.config.spatial.near_distance_m;ds=exp_distance_score(d,scale);dirs=1.;b=None
                if desired is not None:
                    c=subject.centroid;ag=nearest_points(c,g)[1];b=bearing_deg(ag.x-c.x,ag.y-c.y);diff=angular_difference(b,desired);dirs=max(0.,1.-diff/self.config.spatial.directional_tolerance_deg)
                vals.append(.72*ds+.28*dirs);ev.append({'boundary_distance_m':float(d),'bearing':b})
            if not vals:
                return {'evaluated':False,'score':0.,'reason':'anchor has no usable line/polygon geometry for boundary-side test'}
            return {'evaluated':True,'score':float(np.mean(vals)),'evidence':ev}
        if rel=='at_junction_of':
            roads=[a.geometry for a in anchors if a.geometry is not None]
            if len(roads)<2:return {'evaluated':False,'score':0.,'reason':'junction needs >=2 road geometries'}
            points=[]
            for i in range(len(roads)):
                for j in range(i+1,len(roads)):
                    inter=roads[i].intersection(roads[j])
                    if not inter.is_empty:points.append(inter.centroid)
            if not points:return {'evaluated':True,'score':0.,'reason':'resolved roads do not intersect'}
            d=min(subject.distance(p) for p in points);return {'evaluated':True,'score':exp_distance_score(d,self.config.spatial.junction_distance_m),'distance_m':float(d)}
        if rel=='part_of':
            same=any((subject.equals(g) or subject.contains(g) or g.contains(subject)) for g in flat if g.geom_type in ('Polygon','MultiPolygon'));return {'evaluated':True,'score':1. if same else 0.}
        return {'evaluated':False,'score':0.,'reason':f'unsupported relation {rel}'}
    def score_candidate(self,subject,parsed:ParsedDescription,h:GeoHypothesis,local:gpd.GeoDataFrame)->dict:
        evidence=[];weighted=0.;evalw=0.;totalw=0.;reqfail=False
        for c in parsed.constraints:
            r=self.evaluate(subject,c,h,local);w=self.config.spatial.relation_weights.get(c.relation,1.)*constraint_strength_multiplier(c.strength);totalw+=w
            if r['evaluated']:
                evalw+=w;weighted+=r['score']*w
                if c.strength=='required' and r['score']<self.config.spatial.required_constraint_min_score:reqfail=True
            evidence.append({'relation':c.relation,'anchor_ids':c.anchor_ids,'direction':c.direction,'strength':c.strength,'original_phrase':c.original_phrase,**r})
        raw=weighted/evalw if evalw else 0.;coverage=evalw/totalw if totalw else 0.;score=raw*(.55+.45*coverage)
        if reqfail:score*=self.config.spatial.required_constraint_penalty
        return {'spatial_score':float(score),'raw_spatial_score':float(raw),'constraint_coverage':float(coverage),'required_constraint_failure':bool(reqfail),'constraint_evidence':evidence}

## Semantic / Result

In [ ]:
# --------------------------- SEMANTIC / RESULT ---------------------------
class SemanticAdjudicator:
    def __init__(self,client:OpenAI,config:AppConfig):self.client=client;self.config=config
    def assess(self,description:str,parsed:ParsedDescription,candidates:list[dict])->SemanticAssessment:
        payload={'original_description':description,'parsed':parsed.model_dump(),'candidates':[{k:c[k] for k in ['inspire_id','spatial_score','hypothesis_score','constraint_coverage','required_constraint_failure','constraint_evidence']} for c in candidates[:self.config.decision.max_candidates_for_llm]]}
        r=self.client.responses.parse(model=self.config.azure_deployment,instructions=SEMANTIC_PROMPT,input=json.dumps(payload,default=str),text_format=SemanticAssessment);return r.output_parsed

class PipelineResult(BaseModel):
    model_config=ConfigDict(extra='allow')
    record_id:str;status:Literal['MATCHED','MULTIPLE_CANDIDATES','UNRESOLVED','INVALID_INPUT','ERROR'];confidence:float=0.;best_candidate:Optional[dict]=None;candidates:list[dict]=[];parsed_description:Optional[dict]=None;anchor_resolution:list[dict]=[];hypotheses:list[dict]=[];warnings:list[str]=[];errors:list[str]=[]

class LandGeolocationPipeline:
    def __init__(self,config:AppConfig):
        self.config=config;self.azure=OpenAI(base_url=config.azure_endpoint.rstrip('/')+'/openai/v1/',api_key=config.azure_api_key);self.parser=DescriptionParser(self.azure,config);self.uprn_repo=OpenUPRNRepository(config);self.address_repo=AddressRepository(config,self.uprn_repo);self.name_repo=NameRepository(config);self.road_repo=RoadRepository(config);self.inspire_repo=InspireRepository(config);self.anchor_resolver=AnchorResolver(config,self.address_repo,self.name_repo,self.road_repo);self.hypothesis_builder=HypothesisBuilder(config);self.spatial_scorer=SpatialScorer(config,self.road_repo);self.semantic=SemanticAdjudicator(self.azure,config)
    @staticmethod
    def _serialise_anchor_resolution(resolved):return [{'anchor':x.anchor.model_dump(),'candidates':[{'candidate_id':c.candidate_id,'source':c.source,'label':c.label,'match_score':c.match_score,'easting':c.point.x,'northing':c.point.y,'uprns':c.uprns,'member_labels':c.member_labels} for c in x.candidates]} for x in resolved]
    @staticmethod
    def _serialise_hypothesis(h):return {'score':h.score,'lexical_score':h.lexical_score,'coherence_score':h.coherence_score,'assignments':{aid:{'label':c.label,'easting':c.point.x,'northing':c.point.y,'match_score':c.match_score,'uprns':c.uprns} for aid,c in h.assignments.items()}}
    @staticmethod
    def _serialise_candidate(c):
        out={k:v for k,v in c.items() if k!='geometry'}
        if c.get('geometry') is not None:out['geometry_geojson']=mapping(c['geometry'])
        return out
    @staticmethod
    def _anchor_ratio(parsed,h):return sum(1 for a in parsed.anchors if a.id in h.assignments)/len(parsed.anchors) if parsed.anchors else 0.
    def _combine_scores(self,candidates,semantic):
        sm={x.inspire_id:x for x in semantic.candidates} if semantic else {}
        for c in candidates:
            s=sm.get(c['inspire_id']);sem=s.consistency if s else c['spatial_score'];sem*=.4 if s and s.serious_contradiction else 1.;c['semantic_score']=float(sem);c['semantic_explanation']=s.explanation if s else None;c['final_score']=float(self.config.decision.spatial_weight*c['spatial_score']+self.config.decision.hypothesis_weight*c['hypothesis_score']+self.config.decision.semantic_weight*sem)
        return sorted(candidates,key=lambda x:x['final_score'],reverse=True)
    def _decide(self,ranked):
        if not ranked:return 'UNRESOLVED',0.
        b=ranked[0];second=ranked[1]['final_score'] if len(ranked)>1 else 0.;margin=b['final_score']-second
        if b['final_score']>=self.config.decision.matched_threshold and margin>=self.config.decision.minimum_margin and b['constraint_coverage']>=self.config.decision.minimum_constraint_coverage_for_match and b['anchor_resolution_ratio']>=self.config.decision.minimum_anchor_resolution_for_match and not b['required_constraint_failure']:return 'MATCHED',float(b['final_score'])
        if b['final_score']>=self.config.decision.multiple_threshold:return 'MULTIPLE_CANDIDATES',float(b['final_score'])
        return 'UNRESOLVED',float(b['final_score'])
    def process(self,description:str,record_id:Optional[str]=None)->PipelineResult:
        rid=record_id or uuid.uuid4().hex
        if not description or not description.strip():return PipelineResult(record_id=rid,status='INVALID_INPUT',warnings=['Description is empty'])
        warnings=[]
        try:
            parsed=self.parser.parse(description)
            if not parsed.anchors:return PipelineResult(record_id=rid,status='UNRESOLVED',parsed_description=parsed.model_dump(),warnings=['No geographic anchors extracted'])
            resolved=self.anchor_resolver.resolve_all(parsed);unresolved=sum(1 for x in resolved if not x.candidates)
            if unresolved:warnings.append(f'{unresolved} anchor(s) have no OS candidate')
            hypotheses=self.hypothesis_builder.build(resolved)
            if not hypotheses:return PipelineResult(record_id=rid,status='UNRESOLVED',parsed_description=parsed.model_dump(),anchor_resolution=self._serialise_anchor_resolution(resolved),warnings=warnings+['No coherent geographic hypothesis'])
            allc={}
            for hno,h in enumerate(hypotheses[:self.config.decision.max_hypotheses_to_score]):
                bounds=hypothesis_search_bbox(h,parsed,self.config);local=self.inspire_repo.in_bbox(bounds)
                if local.empty:continue
                ratio=self._anchor_ratio(parsed,h)
                for _,row in local.iterrows():
                    geom=row.geometry;s=self.spatial_scorer.score_candidate(geom,parsed,h,local);iid=str(row['inspire_id']);cand={'inspire_id':iid,'geometry':geom,'centroid_easting':float(geom.centroid.x),'centroid_northing':float(geom.centroid.y),'area_m2':float(geom.area),**s,'hypothesis_score':float(h.score),'hypothesis_no':hno,'anchor_resolution_ratio':float(ratio)};pre=.78*cand['spatial_score']+.22*cand['hypothesis_score'];prev=allc.get(iid);prevpre=.78*prev['spatial_score']+.22*prev['hypothesis_score'] if prev else -1
                    if pre>prevpre:allc[iid]=cand
            candidates=sorted(allc.values(),key=lambda x:.78*x['spatial_score']+.22*x['hypothesis_score'],reverse=True)
            if not candidates:return PipelineResult(record_id=rid,status='UNRESOLVED',parsed_description=parsed.model_dump(),anchor_resolution=self._serialise_anchor_resolution(resolved),hypotheses=[self._serialise_hypothesis(h) for h in hypotheses[:10]],warnings=warnings+['No HMLR INSPIRE candidate near plausible hypotheses'])
            semantic=None
            if self.config.llm.use_semantic_adjudicator:
                try:semantic=self.semantic.assess(description,parsed,candidates)
                except Exception as e:warnings.append(f'Semantic adjudication failed; deterministic evidence retained: {e}')
            ranked=self._combine_scores(candidates,semantic);status,confidence=self._decide(ranked);warnings.append('HMLR INSPIRE is indicative registered-freehold extent data, not a definitive legal title boundary.');serial=[self._serialise_candidate(c) for c in ranked[:20]]
            return PipelineResult(record_id=rid,status=status,confidence=confidence,best_candidate=serial[0] if serial else None,candidates=serial,parsed_description=parsed.model_dump(),anchor_resolution=self._serialise_anchor_resolution(resolved),hypotheses=[self._serialise_hypothesis(h) for h in hypotheses[:10]],warnings=warnings)
        except Exception as e:
            logger.exception('Pipeline failed %s',rid);return PipelineResult(record_id=rid,status='ERROR',errors=[str(e)],warnings=warnings)

## Batch / Validation

In [ ]:
# --------------------------- BATCH / VALIDATION ---------------------------
def process_dataframe(df:pd.DataFrame,pipeline:LandGeolocationPipeline,output_prefix:Optional[str]=None)->pd.DataFrame:
    rows=[]
    for _,r in df.iterrows():
        rid=str(r.get('record_id') or uuid.uuid4().hex);res=pipeline.process(str(r.get('description') or ''),rid)
        if output_prefix:write_json_s3(res.model_dump(),output_prefix.rstrip('/')+f'/{rid}.json',s3_client_for(pipeline.config))
        rows.append({'record_id':rid,'status':res.status,'confidence':res.confidence,'best_inspire_id':res.best_candidate.get('inspire_id') if res.best_candidate else None,'candidate_count':len(res.candidates),'warning_count':len(res.warnings),'error_count':len(res.errors)})
    return pd.DataFrame(rows)

def validate_processed_data(config:AppConfig)->dict:
    client=s3_client_for(config);req={'address_records':config.processed.address_records,'address_tokens':config.processed.address_tokens,'name_records':config.processed.name_records,'name_tokens':config.processed.name_tokens,'uprn_records':config.processed.uprn_records,'roads':config.processed.road_geometries,'hmlr_inspire':config.processed.inspire};checks={}
    for name,p in req.items():
        files=list_s3_files(p,('.parquet',),client);checks[name]={'ok':bool(files),'file_count':len(files),'prefix':p}
    checks['all_ok']=all(x['ok'] for x in checks.values());return checks

## Offline Tests

In [ ]:
# --------------------------- OFFLINE TESTS ---------------------------
def run_offline_tests()->dict:
    assert normalize_text(" St. Mary's—Road ")=='st mary s road'
    vs=address_query_variants('10-12 High Street');assert '10 High Street' in vs and '11 High Street' in vs and '12 High Street' in vs
    cfg=AppConfig();builder=HypothesisBuilder(cfg)
    a1=AnchorSpec(id='a1',type='address',raw_text='Rose Cottage',canonical_query='Rose Cottage',search_variants=['Rose Cottage'],may_represent_multiple_properties=False,extraction_confidence=1.)
    a2=AnchorSpec(id='a2',type='road',raw_text='Mill Lane',canonical_query='Mill Lane',search_variants=['Mill Lane'],may_represent_multiple_properties=False,extraction_confidence=1.)
    resolved=[ResolvedAnchorSet(a1,[AnchorCandidate('near','a1','test','near',Point(1000,1000),90),AnchorCandidate('far','a1','test','far',Point(100000,100000),95)]),ResolvedAnchorSet(a2,[AnchorCandidate('road','a2','test','road',Point(1100,1000),88)])]
    hs=builder.build(resolved);assert hs and hs[0].assignments['a1'].candidate_id=='near'
    class FakeRoad:
        def nearest_road(self,p,radius_m=150):return LineString([(p.x-100,p.y-20),(p.x+100,p.y-20)])
    scorer=SpatialScorer(cfg,FakeRoad());parsed=ParsedDescription(subject_type='land',anchors=[a1,a2],constraints=[SpatialConstraint(relation='between',anchor_ids=['a1','a2'],strength='strong',original_phrase='between')],normalized_description='test',ambiguity_notes=[])
    h=GeoHypothesis({'a1':AnchorCandidate('1','a1','test','A',Point(0,0),90,uprns=['u1']),'a2':AnchorCandidate('2','a2','test','B',Point(100,0),90)},.9,.9,.9);local=gpd.GeoDataFrame({'inspire_id':['x'],'geometry':[box(40,-10,60,10)]},geometry='geometry',crs=27700);assert scorer.score_candidate(local.iloc[0].geometry,parsed,h,local)['spatial_score']>.7
    adj=parsed.model_copy(update={'constraints':[SpatialConstraint(relation='adjacent_to',anchor_ids=['a1'],strength='required',original_phrase='adjacent')]});anchor_poly=box(-10,-10,10,10);adj_poly=box(10,-10,30,10);local2=gpd.GeoDataFrame({'inspire_id':['a','b'],'geometry':[anchor_poly,adj_poly]},geometry='geometry',crs=27700);same=scorer.score_candidate(anchor_poly,adj,h,local2)['spatial_score'];near=scorer.score_candidate(adj_poly,adj,h,local2)['spatial_score'];assert near>same
    stub=object.__new__(LandGeolocationPipeline);stub.config=cfg;status,_=stub._decide([{'final_score':.85,'constraint_coverage':.9,'anchor_resolution_ratio':1.,'required_constraint_failure':False},{'final_score':.82,'constraint_coverage':.9,'anchor_resolution_ratio':1.,'required_constraint_failure':False}]);assert status=='MULTIPLE_CANDIDATES';status,_=stub._decide([{'final_score':.90,'constraint_coverage':.95,'anchor_resolution_ratio':1.,'required_constraint_failure':False},{'final_score':.60,'constraint_coverage':.95,'anchor_resolution_ratio':1.,'required_constraint_failure':False}]);assert status=='MATCHED'
    # Nearby single-property duplicates must stay separate; true range/group anchors may cluster.
    rs=[{'label':'Rose A','easting':0,'northing':0,'match_score':90,'uprns':['1'],'member_labels':['Rose A']},{'label':'Rose B','easting':20,'northing':0,'match_score':89,'uprns':['2'],'member_labels':['Rose B']}]
    assert len(cluster_address_results('single',rs,group_nearby=False))==2
    assert len(cluster_address_results('group',rs,group_nearby=True))==1
    # Direction and rear/front orientation.
    north_parsed=ParsedDescription(subject_type='land',anchors=[a1],constraints=[SpatialConstraint(relation='north_of',anchor_ids=['a1'],strength='strong',original_phrase='north')],normalized_description='x',ambiguity_notes=[])
    north_poly=box(-10,40,10,60); east_poly=box(40,-10,60,10)
    assert scorer.score_candidate(north_poly,north_parsed,h,local)['spatial_score'] > scorer.score_candidate(east_poly,north_parsed,h,local)['spatial_score']
    rear_parsed=ParsedDescription(subject_type='land',anchors=[a1],constraints=[SpatialConstraint(relation='rear_of',anchor_ids=['a1'],strength='strong',original_phrase='rear')],normalized_description='x',ambiguity_notes=[])
    assert scorer.score_candidate(north_poly,rear_parsed,h,local)['spatial_score'] > scorer.score_candidate(box(-10,-60,10,-40),rear_parsed,h,local)['spatial_score']
    return {'ok':True,'tests':['normalization','range expansion','joint ambiguity beam','between','adjacency same-extent safeguard','single-vs-group clustering','north direction','rear/front orientation','conservative decision']}

## Configure S3 + Azure

Set the bucket once. Raw paths are **folder prefixes**, not individual filenames. No release folders are required.

In [ ]:
CONFIG = AppConfig()
BUCKET = "YOUR-BUCKET"

CONFIG.raw.addressbase_core = f"s3://{BUCKET}/raw/os/addressbase_core/"
CONFIG.raw.open_names = f"s3://{BUCKET}/raw/os/open_names/"
CONFIG.raw.open_uprn = f"s3://{BUCKET}/raw/os/open_uprn/"
CONFIG.raw.open_roads = f"s3://{BUCKET}/raw/os/open_roads/"
CONFIG.raw.hmlr_inspire = f"s3://{BUCKET}/raw/hmlr/la/"

CONFIG.processed.address_records = f"s3://{BUCKET}/processed/os/address_records/"
CONFIG.processed.address_tokens = f"s3://{BUCKET}/processed/os/address_tokens/"
CONFIG.processed.name_records = f"s3://{BUCKET}/processed/os/name_records/"
CONFIG.processed.name_tokens = f"s3://{BUCKET}/processed/os/name_tokens/"
CONFIG.processed.uprn_records = f"s3://{BUCKET}/processed/os/uprn_records/"
CONFIG.processed.road_geometries = f"s3://{BUCKET}/processed/os/roads/"
CONFIG.processed.inspire = f"s3://{BUCKET}/processed/hmlr/inspire/"
CONFIG.processed.results = f"s3://{BUCKET}/results/"

# Prefer environment variables for credentials:
# AZURE_OPENAI_ENDPOINT
# AZURE_OPENAI_API_KEY
# AZURE_OPENAI_DEPLOYMENT
# Or set CONFIG.azure_endpoint / api_key / deployment here.

## First run: inspect, test, then preprocess

Do not start the national preprocessing job until raw inspection succeeds.

In [ ]:
# Reads only tiny samples / file listings.
# raw_report = inspect_raw_data(CONFIG)
# print(json.dumps(raw_report, indent=2, default=str))

# Deterministic tests do not call Azure or S3.
print(run_offline_tests())

## Build processed S3 data

`overwrite=True` deletes only the configured **processed** prefixes, never `raw/`. Use it intentionally.

In [ ]:
# preprocessing_report = run_full_preprocessing(CONFIG, overwrite=True)
# print(json.dumps(preprocessing_report, indent=2, default=str))

# processed_report = validate_processed_data(CONFIG)
# print(json.dumps(processed_report, indent=2, default=str))

## Run one complex description

In [ ]:
# pipeline = LandGeolocationPipeline(CONFIG)
# result = pipeline.process(
#     "Land adjoining 12 and 14 High Street, "
#     "lying to the rear of 10-16 High Street "
#     "and bounded on the east by Mill Lane, Chester",
#     record_id="ABC123",
# )
# print(json.dumps(result.model_dump(), indent=2, default=str))

## What to tune after you have labelled examples

Defaults are deliberately conservative. Tune the following using known examples and optimize for **low false-`MATCHED` rate**:

- Address/name match thresholds.
- `beam_width` and geographic coherence scale.
- Single-anchor and multi-anchor search radii.
- Adjacency, between, boundary, rear/front and direction tolerances.
- Relation weights.
- Final `matched_threshold`, `minimum_margin`, anchor coverage and constraint coverage.

The data you currently have does **not** include full river/rail geometry. Named watercourses/railways may be discoverable through Open Names, but a phrase such as “bounded by River X” will be treated as insufficiently verified unless usable line/polygon geometry is available. That is intentional: missing evidence should reduce confidence rather than create a false match.